# 대리 모델 재현 실험 (1단계) — Qwen3-4B-Instruct-2507, 그룹 C

**목적**: 본실험의 Qwen3-30B-A3B-Instruct-2507에서 나타난 "DAP → 모호한 완결 문항(그룹 C)에서 '거짓' 과신" 패턴이
Colab 무료 T4에 올라가는 같은 계열의 소형 모델에서도 재현되는지 확인한다. 재현되면 2단계(logit lens)를 이 모델로 진행한다.

**설계**: 그룹 C 100문항 × {Zero-shot, DAP} = 200회 생성. 프롬프트·파서는 본실험(`prompts.py`, `parser.py`)과 동일. 그리디 디코딩(temperature 0에 해당).

**사전 등록한 판정 기준** (결과를 보기 전에 고정):
- **재현**: FBR_C(DAP) − FBR_C(Zero-shot) ≥ 0.15 **그리고** 같은 문항 쌍에 대한 정확 McNemar 검정 p < 0.05 (거짓 응답 증가 방향)
- **부분 재현**: ACC_C는 유의하게 떨어졌지만 오답이 '거짓'이 아닌 다른 방향인 경우 → 결과는 보고하되 2단계 진행 여부는 다시 판단
- **미재현**: 그 외 → 메커니즘 분석은 향후 과제로 넘기고, "모델 크기에 따라 달라지는 현상"으로 보고

참고(본실험 Qwen3-30B, 그룹 C): Zero-shot ACC 0.80 / TBR 0.15 / FBR 0.05, DAP ACC 0.47 / TBR 0.11 / FBR 0.42

**실행 방법**: 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후, 위에서부터 순서대로 실행. 전체 약 10~15분.
세션이 끊겨도 결과가 한 줄씩 저장되므로, 처음부터 다시 실행하면 이미 끝난 문항은 건너뛴다(Google Drive 저장 사용 시).

## 0. 설정

In [ ]:
# ===== 설정 (필요하면 여기만 수정) =====
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"   # 대안: "Qwen/Qwen3-8B" (+ LOAD_MODE="4bit")
LOAD_MODE = "fp16"                          # "fp16" 또는 "4bit". fp16에서 출력이 깨지면 "4bit"로
MODEL_KEY = "qwen4b_proxy"                  # 결과 파일의 model 필드 값
USE_DRIVE = True                            # True면 Google Drive에 저장(세션 끊겨도 이어서 실행 가능)
MAX_NEW_TOKENS = 16
STRATEGIES = ["zero-shot", "dap"]

## 1. GPU 확인 및 패키지 설치

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

## 2. 저장 위치

In [ ]:
import os, json
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/imperfective_proxy'
else:
    OUT_DIR = '/content/imperfective_proxy'
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, f'{MODEL_KEY}_groupC.jsonl')
print('결과 파일:', OUT_PATH)

## 3. 데이터 (그룹 C 100문항, 본실험과 동일)

In [ ]:
ITEMS_C = json.loads(r'''[
{
"id": "C_001",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "고치다",
"premise": "그는 고장 난 시계를 고치고 있었다.",
"hypothesis": "그는 고장 난 시계를 다 고쳤다."
},
{
"id": "C_002",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "채우다",
"premise": "그녀는 컵에 물을 채우고 있었다.",
"hypothesis": "그녀는 컵에 물을 가득 채웠다."
},
{
"id": "C_003",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "청소하다",
"premise": "그는 방 하나를 청소하고 있었다.",
"hypothesis": "그는 방 하나 전체를 청소했다."
},
{
"id": "C_004",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "수리하다",
"premise": "정비사가 차 한 대를 수리하고 있었다.",
"hypothesis": "정비사가 차 한 대를 다 수리했다."
},
{
"id": "C_005",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "비우다",
"premise": "그는 서랍을 비우고 있었다.",
"hypothesis": "그는 서랍을 다 비웠다."
},
{
"id": "C_006",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "말리다",
"premise": "그녀는 빨래를 말리고 있었다.",
"hypothesis": "그녀는 빨래를 완전히 말렸다."
},
{
"id": "C_007",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "세탁하다",
"premise": "그는 옷 한 벌을 세탁하고 있었다.",
"hypothesis": "그는 옷 한 벌을 다 세탁했다."
},
{
"id": "C_008",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "다림질하다",
"premise": "그녀는 셔츠 한 장을 다림질하고 있었다.",
"hypothesis": "그녀는 셔츠 한 장을 다 다림질했다."
},
{
"id": "C_009",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "깎다",
"premise": "그는 연필을 깎고 있었다.",
"hypothesis": "그는 연필을 다 깎았다."
},
{
"id": "C_010",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "부풀리다",
"premise": "그는 풍선을 최대한 부풀리고 있었다.",
"hypothesis": "그는 풍선을 최대한 다 부풀렸다."
},
{
"id": "C_011",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "얼리다",
"premise": "그녀는 물을 얼리고 있었다.",
"hypothesis": "그녀는 물을 다 얼렸다."
},
{
"id": "C_012",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "녹이다",
"premise": "그는 버터를 녹이고 있었다.",
"hypothesis": "그는 버터를 다 녹였다."
},
{
"id": "C_013",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "봉합하다",
"premise": "의사가 상처를 봉합하고 있었다.",
"hypothesis": "의사가 상처를 다 봉합했다."
},
{
"id": "C_014",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "개조하다",
"premise": "그들은 집을 개조하고 있었다.",
"hypothesis": "그들은 집을 다 개조했다."
},
{
"id": "C_015",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "철거하다",
"premise": "인부들이 건물을 철거하고 있었다.",
"hypothesis": "인부들이 건물을 다 철거했다."
},
{
"id": "C_016",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "파다",
"premise": "그는 구덩이 하나를 파고 있었다.",
"hypothesis": "그는 구덩이 하나를 다 팠다."
},
{
"id": "C_017",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "싸다",
"premise": "그녀는 짐을 싸고 있었다.",
"hypothesis": "그녀는 짐을 다 쌌다."
},
{
"id": "C_018",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "풀다",
"premise": "그는 매듭 하나를 풀고 있었다.",
"hypothesis": "그는 매듭 하나를 다 풀었다."
},
{
"id": "C_019",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "펴다",
"premise": "그녀는 구겨진 종이를 펴고 있었다.",
"hypothesis": "그녀는 구겨진 종이를 다 폈다."
},
{
"id": "C_020",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "해독하다",
"premise": "그는 암호 하나를 해독하고 있었다.",
"hypothesis": "그는 암호 하나를 다 해독했다."
},
{
"id": "C_021",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "정리하다",
"premise": "그는 책상 하나를 정리하고 있었다.",
"hypothesis": "그는 책상 하나를 다 정리했다."
},
{
"id": "C_022",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "지우다",
"premise": "그녀는 칠판을 지우고 있었다.",
"hypothesis": "그녀는 칠판을 전부 지웠다."
},
{
"id": "C_023",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "수정하다",
"premise": "그는 보고서를 수정하고 있었다.",
"hypothesis": "그는 보고서를 다 수정했다."
},
{
"id": "C_024",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "설치하다",
"premise": "그녀는 프로그램을 설치하고 있었다.",
"hypothesis": "그녀는 프로그램을 다 설치했다."
},
{
"id": "C_025",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "포장하다",
"premise": "그는 선물을 포장하고 있었다.",
"hypothesis": "그는 선물을 다 포장했다."
},
{
"id": "C_026",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "제거하다",
"premise": "그녀는 자전거의 녹을 제거하고 있었다.",
"hypothesis": "그녀는 자전거의 녹을 다 제거했다."
},
{
"id": "C_027",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "메우다",
"premise": "그는 벽의 틈을 메우고 있었다.",
"hypothesis": "그는 벽의 틈을 다 메웠다."
},
{
"id": "C_028",
"verb_class": "Change_of_State",
"secondary_class": "전형적 달성동사",
"verb": "구조하다",
"premise": "소방관이 아이를 구조하고 있었다.",
"hypothesis": "소방관이 아이를 다 구조했다."
},
{
"id": "C_029",
"verb_class": "Change_of_State",
"secondary_class": "전형적 달성동사",
"verb": "구해내다",
"premise": "그는 물에 빠진 사람을 구해내고 있었다.",
"hypothesis": "그는 물에 빠진 사람을 다 구해냈다."
},
{
"id": "C_030",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "다듬다",
"premise": "그는 잔디밭을 다듬고 있었다.",
"hypothesis": "그는 잔디밭 전체를 다듬었다."
},
{
"id": "C_031",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "쓸다",
"premise": "그녀는 바닥을 쓸고 있었다.",
"hypothesis": "그녀는 바닥 전체를 쓸었다."
},
{
"id": "C_032",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "닦다",
"premise": "그는 복도를 대걸레로 닦고 있었다.",
"hypothesis": "그는 복도 전체를 대걸레로 닦았다."
},
{
"id": "C_033",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "광내다",
"premise": "그녀는 구두를 광내고 있었다.",
"hypothesis": "그녀는 구두를 다 광냈다."
},
{
"id": "C_034",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "사포질하다",
"premise": "그는 책상을 사포질하고 있었다.",
"hypothesis": "그는 책상 전체를 사포질했다."
},
{
"id": "C_035",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "없애다",
"premise": "그녀는 욕실의 곰팡이를 없애고 있었다.",
"hypothesis": "그녀는 욕실의 곰팡이를 다 없앴다."
},
{
"id": "C_036",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "자르다",
"premise": "그는 종이 한 장을 자르고 있었다.",
"hypothesis": "그는 종이 한 장을 전부 잘랐다."
},
{
"id": "C_037",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "썰다",
"premise": "그녀는 양파 하나를 얇게 썰고 있었다.",
"hypothesis": "그녀는 양파 하나를 얇게 다 썰었다."
},
{
"id": "C_038",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "다지다",
"premise": "그는 마늘을 다지고 있었다.",
"hypothesis": "그는 마늘을 다 다졌다."
},
{
"id": "C_039",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "끓이다",
"premise": "그녀는 물을 끓이고 있었다.",
"hypothesis": "그녀는 물을 다 끓였다."
},
{
"id": "C_040",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "튀기다",
"premise": "그는 감자를 튀기고 있었다.",
"hypothesis": "그는 감자를 다 튀겼다."
},
{
"id": "C_041",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "굽다",
"premise": "그녀는 고기를 다 익을 때까지 굽고 있었다.",
"hypothesis": "그녀는 고기를 다 구웠다."
},
{
"id": "C_042",
"verb_class": "Change_of_State",
"secondary_class": "정도달성동사",
"verb": "데우다",
"premise": "그는 국 한 그릇을 데우고 있었다.",
"hypothesis": "그는 국 한 그릇을 다 데웠다."
},
{
"id": "C_043",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "장식하다",
"premise": "그녀는 케이크를 장식하고 있었다.",
"hypothesis": "그녀는 케이크를 다 장식했다."
},
{
"id": "C_044",
"verb_class": "Change_of_State",
"secondary_class": "완성동사",
"verb": "들이다",
"premise": "그들은 거실에 가구를 들이고 있었다.",
"hypothesis": "그들은 거실에 가구를 다 들였다."
},
{
"id": "C_045",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "먹다",
"premise": "그는 밥 한 그릇을 먹고 있었다.",
"hypothesis": "그는 밥 한 그릇을 다 먹었다."
},
{
"id": "C_046",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "마시다",
"premise": "그녀는 물 한 잔을 마시고 있었다.",
"hypothesis": "그녀는 물 한 잔을 다 마셨다."
},
{
"id": "C_047",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "읽다",
"premise": "그는 책 한 권을 읽고 있었다.",
"hypothesis": "그는 책 한 권을 다 읽었다."
},
{
"id": "C_048",
"verb_class": "Consumption",
"secondary_class": "정도달성동사",
"verb": "태우다",
"premise": "그녀는 종이 한 장을 태우고 있었다.",
"hypothesis": "그녀는 종이 한 장을 완전히 태웠다."
},
{
"id": "C_049",
"verb_class": "Consumption",
"secondary_class": "전형적 달성동사",
"verb": "삼키다",
"premise": "그는 약 한 알을 삼키고 있었다.",
"hypothesis": "그는 약 한 알을 다 삼켰다."
},
{
"id": "C_050",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "소모하다",
"premise": "차가 연료 1리터를 소모하고 있었다.",
"hypothesis": "차가 연료 1리터를 전부 소모했다."
},
{
"id": "C_051",
"verb_class": "Consumption",
"secondary_class": "전형적 달성동사",
"verb": "부수다",
"premise": "그들은 모래성 하나를 부수고 있었다.",
"hypothesis": "그들은 모래성 하나를 모조리 부쉈다."
},
{
"id": "C_052",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "해결하다",
"premise": "그는 수학 문제 하나를 해결하고 있었다.",
"hypothesis": "그는 수학 문제 하나를 다 해결했다."
},
{
"id": "C_053",
"verb_class": "Consumption",
"secondary_class": "완성동사",
"verb": "벗기다",
"premise": "그녀는 사과 하나의 껍질을 벗기고 있었다.",
"hypothesis": "그녀는 사과 하나의 껍질을 다 벗겼다."
},
{
"id": "C_054",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "짓다",
"premise": "그는 정자를 짓고 있었다.",
"hypothesis": "그는 정자를 다 지었다."
},
{
"id": "C_055",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "칠하다",
"premise": "그녀는 벽 하나를 칠하고 있었다.",
"hypothesis": "그녀는 벽 하나를 다 칠했다."
},
{
"id": "C_056",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "새기다",
"premise": "그는 목판에 글자를 새기고 있었다.",
"hypothesis": "그는 목판에 글자를 다 새겼다."
},
{
"id": "C_057",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "쓰다",
"premise": "그녀는 편지 한 통을 쓰고 있었다.",
"hypothesis": "그녀는 편지 한 통을 다 썼다."
},
{
"id": "C_058",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "뜨개질하다",
"premise": "그는 스웨터 한 벌을 뜨개질하고 있었다.",
"hypothesis": "그는 스웨터 한 벌을 다 뜨개질했다."
},
{
"id": "C_059",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "작곡하다",
"premise": "그녀는 곡 하나를 작곡하고 있었다.",
"hypothesis": "그녀는 곡 하나를 다 작곡했다."
},
{
"id": "C_060",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "그리다",
"premise": "그는 그림 한 점을 그리고 있었다.",
"hypothesis": "그는 그림 한 점을 다 그렸다."
},
{
"id": "C_061",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "촬영하다",
"premise": "그들은 영화 한 편을 촬영하고 있었다.",
"hypothesis": "그들은 영화 한 편을 다 촬영했다."
},
{
"id": "C_062",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "코딩하다",
"premise": "그는 프로그램 하나를 코딩하고 있었다.",
"hypothesis": "그는 프로그램 하나를 다 코딩했다."
},
{
"id": "C_063",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "바느질하다",
"premise": "그녀는 옷 한 벌을 바느질하고 있었다.",
"hypothesis": "그녀는 옷 한 벌을 다 바느질했다."
},
{
"id": "C_064",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "조각하다",
"premise": "그는 조각상 하나를 조각하고 있었다.",
"hypothesis": "그는 조각상 하나를 다 조각했다."
},
{
"id": "C_065",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "짜다",
"premise": "그녀는 천 한 필을 짜고 있었다.",
"hypothesis": "그녀는 천 한 필을 다 짰다."
},
{
"id": "C_066",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "양조하다",
"premise": "그는 맥주 한 통을 양조하고 있었다.",
"hypothesis": "그는 맥주 한 통을 다 양조했다."
},
{
"id": "C_067",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "위조하다",
"premise": "그녀는 졸업장 하나를 위조하고 있었다.",
"hypothesis": "그녀는 졸업장 하나를 다 위조했다."
},
{
"id": "C_068",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "설계하다",
"premise": "그는 건물 하나를 설계하고 있었다.",
"hypothesis": "그는 건물 하나를 다 설계했다."
},
{
"id": "C_069",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "증명하다",
"premise": "그녀는 가설 하나를 증명하고 있었다.",
"hypothesis": "그녀는 가설 하나를 다 증명했다."
},
{
"id": "C_070",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "번역하다",
"premise": "그는 소설 한 권을 번역하고 있었다.",
"hypothesis": "그는 소설 한 권을 다 번역했다."
},
{
"id": "C_071",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "건설하다",
"premise": "그들은 다리 하나를 건설하고 있었다.",
"hypothesis": "그들은 다리 하나를 다 건설했다."
},
{
"id": "C_072",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "발명하다",
"premise": "그녀는 기계 하나를 발명하고 있었다.",
"hypothesis": "그녀는 기계 하나를 다 발명했다."
},
{
"id": "C_073",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "생산하다",
"premise": "공장이 자동차 100대를 생산하고 있었다.",
"hypothesis": "공장이 자동차 100대를 생산했다."
},
{
"id": "C_074",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "세우다",
"premise": "그는 여행 계획 하나를 세우고 있었다.",
"hypothesis": "그는 여행 계획 하나를 다 세웠다."
},
{
"id": "C_075",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "계산하다",
"premise": "그녀는 총합을 계산하고 있었다.",
"hypothesis": "그녀는 총합을 다 계산했다."
},
{
"id": "C_076",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "측정하다",
"premise": "그는 방의 넓이를 측정하고 있었다.",
"hypothesis": "그는 방의 넓이를 다 측정했다."
},
{
"id": "C_077",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "세다",
"premise": "그녀는 동전 100개를 세고 있었다.",
"hypothesis": "그녀는 동전 100개를 세었다."
},
{
"id": "C_078",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "받아쓰다",
"premise": "그는 문장 하나를 받아쓰고 있었다.",
"hypothesis": "그는 문장 하나를 다 받아썼다."
},
{
"id": "C_079",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "다운로드하다",
"premise": "그녀는 파일 하나를 다운로드하고 있었다.",
"hypothesis": "그녀는 파일 하나를 다 다운로드했다."
},
{
"id": "C_080",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "업로드하다",
"premise": "그는 영상 하나를 업로드하고 있었다.",
"hypothesis": "그는 영상 하나를 다 업로드했다."
},
{
"id": "C_081",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "인쇄하다",
"premise": "그녀는 문서 한 부를 인쇄하고 있었다.",
"hypothesis": "그녀는 문서 한 부를 다 인쇄했다."
},
{
"id": "C_082",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "심다",
"premise": "그는 나무 한 그루를 심고 있었다.",
"hypothesis": "그는 나무 한 그루를 다 심었다."
},
{
"id": "C_083",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "수확하다",
"premise": "농부가 벼 한 마지기를 수확하고 있었다.",
"hypothesis": "농부가 벼 한 마지기를 다 수확했다."
},
{
"id": "C_084",
"verb_class": "Creation",
"secondary_class": "정도달성동사",
"verb": "섞다",
"premise": "그녀는 재료 세 가지를 섞고 있었다.",
"hypothesis": "그녀는 재료 세 가지를 다 섞었다."
},
{
"id": "C_085",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "갖추다",
"premise": "그들은 실험실에 장비를 갖추고 있었다.",
"hypothesis": "그들은 실험실에 장비를 다 갖췄다."
},
{
"id": "C_086",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "설립하다",
"premise": "그는 회사 하나를 설립하고 있었다.",
"hypothesis": "그는 회사 하나를 다 설립했다."
},
{
"id": "C_087",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "형성하다",
"premise": "그녀는 새로운 습관을 형성하고 있었다.",
"hypothesis": "그녀는 새로운 습관을 다 형성했다."
},
{
"id": "C_088",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "빚다",
"premise": "그는 만두 100개를 빚고 있었다.",
"hypothesis": "그는 만두 100개를 빚었다."
},
{
"id": "C_089",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "조립하다",
"premise": "그녀는 가구 하나를 조립하고 있었다.",
"hypothesis": "그녀는 가구 하나를 다 조립했다."
},
{
"id": "C_090",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "모으다",
"premise": "그는 동전 100개를 모으고 있었다.",
"hypothesis": "그는 동전 100개를 모았다."
},
{
"id": "C_091",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "수집하다",
"premise": "그녀는 우표 50장을 수집하고 있었다.",
"hypothesis": "그녀는 우표 50장을 수집했다."
},
{
"id": "C_092",
"verb_class": "Creation",
"secondary_class": "완성동사",
"verb": "따다",
"premise": "그는 사과 10개를 따고 있었다.",
"hypothesis": "그는 사과 10개를 땄다."
},
{
"id": "C_093",
"verb_class": "Motion_to_Goal",
"secondary_class": "전형적 달성동사",
"verb": "오르다",
"premise": "그는 산 정상에 오르고 있었다.",
"hypothesis": "그는 산 정상에 올랐다."
},
{
"id": "C_094",
"verb_class": "Motion_to_Goal",
"secondary_class": "완성동사",
"verb": "건너다",
"premise": "그녀는 강을 건너고 있었다.",
"hypothesis": "그녀는 강을 건넜다."
},
{
"id": "C_095",
"verb_class": "Motion_to_Goal",
"secondary_class": "완성동사",
"verb": "등산하다",
"premise": "그는 정상까지 등산하고 있었다.",
"hypothesis": "그는 정상까지 등산했다."
},
{
"id": "C_096",
"verb_class": "Motion_to_Goal",
"secondary_class": "완성동사",
"verb": "날아가다",
"premise": "비행기가 서울까지 날아가고 있었다.",
"hypothesis": "비행기가 서울까지 날아갔다."
},
{
"id": "C_097",
"verb_class": "Motion_to_Goal",
"secondary_class": "완성동사",
"verb": "배달하다",
"premise": "그는 피자 한 판을 배달하고 있었다.",
"hypothesis": "그는 피자 한 판을 배달했다."
},
{
"id": "C_098",
"verb_class": "Motion_to_Goal",
"secondary_class": "전형적 달성동사",
"verb": "도착하다",
"premise": "비행기가 공항에 도착하고 있었다.",
"hypothesis": "비행기가 목적지 공항에 도착했다."
},
{
"id": "C_099",
"verb_class": "Motion_to_Goal",
"secondary_class": "전형적 달성동사",
"verb": "입장하다",
"premise": "그녀는 경기장에 입장하고 있었다.",
"hypothesis": "그녀는 경기장에 입장했다."
},
{
"id": "C_100",
"verb_class": "Motion_to_Goal",
"secondary_class": "전형적 달성동사",
"verb": "도달하다",
"premise": "그는 결승지점에 도달하고 있었다.",
"hypothesis": "그는 결승지점에 도달했다."
}
]''')
print(len(ITEMS_C), '문항 |', ITEMS_C[0])

## 4. 프롬프트와 파서 (본실험 `prompts.py`, `parser.py`와 동일)

In [ ]:
PROMPTS = {
    "zero-shot": {
        "system": """당신은 엄격한 논리학자입니다. 당신의 임무는 전제(Premise)가 주어졌을 때 가설(Hypothesis)이 반드시 참인지 판단하는 것입니다.
전제만으로 가설이 반드시 참이면 "참"을 출력하십시오.
가설이 전제와 모순되면 "거짓"을 출력하십시오.
가설이 참일 수도 있지만 전제에서 명시적으로 보장되지 않으면 "모름"을 출력하십시오.
상식적 추측을 사용하지 마십시오. 오직 제공된 텍스트만 사용하십시오.""",
        "user_format": """입력: 전제: {premise} 가설: {hypothesis}
다음 중 하나로만 답하십시오: "참", "거짓", 또는 "모름".""",
    },
    "dap": {
        "system": """당신은 엄격한 논리학자입니다. 당신의 임무는 전제(Premise)가 주어졌을 때 가설(Hypothesis)이 반드시 참인지 판단하는 것입니다.
전제만으로 가설이 반드시 참이면 "참"을 출력하십시오.
가설이 전제와 모순되면 "거짓"을 출력하십시오.
가설이 참일 수도 있지만 전제에서 명시적으로 보장되지 않으면 "모름"을 출력하십시오.

중요한 언어학적 규칙: 활동동사(activity)와 완수동사(accomplishment)의 차이에 주의하십시오.
- 완수동사/목표지향적 행위(예: "짓고 있었다", "쓰고 있었다")의 경우, 진행형이 완료를 의미하지 않습니다.
- 활동동사(예: "달리고 있었다", "노래하고 있었다")의 경우, 그 과정 자체가 행동이 일어났음을 의미합니다.
상식적 추측을 사용하지 마십시오. 오직 제공된 텍스트만 사용하십시오.""",
        "user_format": """입력: 전제: {premise} 가설: {hypothesis}
다음 중 하나로만 답하십시오: "참", "거짓", 또는 "모름".""",
    },
}

LABEL_MAP = {"참": "True", "거짓": "False", "모름": "Unknown"}

def parse_simple(raw_text):
    text = (raw_text or "").strip()
    found = [ko for ko in LABEL_MAP if ko in text]
    if len(found) == 1:
        return {"label": LABEL_MAP[found[0]], "parse_ok": True}
    return {"label": None, "parse_ok": False}

assert parse_simple("참")["label"] == "True" and parse_simple('"모름"')["label"] == "Unknown"
assert parse_simple("참 또는 거짓")["parse_ok"] is False

## 5. 모델 불러오기

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if LOAD_MODE == "4bit":
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=qcfg, device_map="cuda")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
model.eval()
print('레이어 수:', model.config.num_hidden_layers, '| GPU 메모리 사용:', round(torch.cuda.memory_allocated()/1e9, 1), 'GB')

# 라벨 단어의 첫 토큰 id (2단계 logit lens에서도 같은 값을 씀)
LABEL_TOKENS = {}
for ko in LABEL_MAP:
    ids = tok.encode(ko, add_special_tokens=False)
    LABEL_TOKENS[ko] = ids[0]
    print(f'{ko}: 토큰 {ids} -> {[tok.decode([i]) for i in ids]}')
assert len(set(LABEL_TOKENS.values())) == 3, '세 라벨의 첫 토큰이 겹칩니다 — 2단계 전에 토큰 처리 방식을 바꿔야 합니다.'

## 6. 생성 함수 + 동작 확인 (2문항)

In [ ]:
@torch.no_grad()
def run_one(strategy, premise, hypothesis):
    p = PROMPTS[strategy]
    messages = [{"role": "system", "content": p["system"]},
                {"role": "user", "content": p["user_format"].format(premise=premise, hypothesis=hypothesis)}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)  # 2507 Instruct에서는 무시됨, Qwen3-8B 대비용
    enc = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         output_scores=True, return_dict_in_generate=True,
                         pad_token_id=tok.eos_token_id)
    gen_ids = out.sequences[0, enc.input_ids.shape[1]:]
    raw = tok.decode(gen_ids, skip_special_tokens=True)
    # 첫 생성 위치에서 세 라벨 토큰의 확률 (전체 어휘 softmax 기준)
    probs = torch.softmax(out.scores[0][0].float(), dim=-1)
    label_probs = {LABEL_MAP[ko]: round(probs[t].item(), 5) for ko, t in LABEL_TOKENS.items()}
    first_tok = tok.decode(gen_ids[:1])
    return raw, label_probs, first_tok

for it in ITEMS_C[:2]:
    for s in STRATEGIES:
        raw, lp, ft = run_one(s, it["premise"], it["hypothesis"])
        print(s, '|', it["premise"], '->', repr(raw), '| 첫 토큰', repr(ft), '|', lp)
print('\n위 출력이 "참/거짓/모름" 중 하나로 나오면 정상. 깨진 문자나 빈 출력이면 LOAD_MODE="4bit"로 바꿔 5번부터 다시 실행.')

## 7. 본 실행 (200회, 한 줄씩 저장 · 이어서 실행 가능)

In [ ]:
import time
done = set()
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r = json.loads(line); done.add((r['strategy'], r['id']))
print('이미 완료:', len(done))

t0 = time.time(); n_new = 0
with open(OUT_PATH, 'a', encoding='utf-8') as f:
    for s in STRATEGIES:
        for it in ITEMS_C:
            if (s, it['id']) in done:
                continue
            raw, lp, ft = run_one(s, it['premise'], it['hypothesis'])
            parsed = parse_simple(raw)
            rec = dict(id=it['id'], group='C_Ambiguous_Accomplishment', verb_class=it['verb_class'],
                       secondary_class=it['secondary_class'], verb=it['verb'],
                       premise=it['premise'], hypothesis=it['hypothesis'], gold_label='Unknown',
                       model=MODEL_KEY, model_name=MODEL_ID, strategy=s,
                       predicted_label=parsed['label'], parse_ok=parsed['parse_ok'],
                       raw_response=raw, reasoning=None, possible_interruptions=None,
                       first_token=ft, label_probs=lp, load_mode=LOAD_MODE)
            f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
            n_new += 1
            if n_new % 20 == 0:
                print(f'{n_new}건 완료 ({time.time()-t0:.0f}초)')
print(f'끝. 새로 {n_new}건, {time.time()-t0:.0f}초')

## 8. 지표 계산 및 사전 등록 기준 판정

In [ ]:
import pandas as pd
from scipy.stats import binomtest

df = pd.read_json(OUT_PATH, lines=True)
df = df[df.strategy.isin(STRATEGIES)]
print('파싱 성공률:', df.groupby('strategy').parse_ok.mean().round(3).to_dict())

def metrics(d):
    ok = d[d.parse_ok]
    return pd.Series({'n': len(d), 'parse_ok': d.parse_ok.mean(),
                      'ACC_C': (ok.predicted_label == 'Unknown').mean(),
                      'TBR_C': (ok.predicted_label == 'True').mean(),
                      'FBR_C': (ok.predicted_label == 'False').mean()})

summary = df.groupby('strategy').apply(metrics).round(3)
ref = pd.DataFrame({'ACC_C': [0.80, 0.47], 'TBR_C': [0.15, 0.11], 'FBR_C': [0.05, 0.42]},
                   index=['zero-shot', 'dap'])
print('\n[대리 모델]'); print(summary)
print('\n[참고: 본실험 Qwen3-30B]'); print(ref)

# 문항 쌍 비교: 같은 문항이 Zero-shot -> DAP에서 '거짓'으로 바뀌었는가 (정확 McNemar)
w = df.pivot_table(index='id', columns='strategy', values='predicted_label', aggfunc='first')
w = w.dropna()
zs_false, dap_false = (w['zero-shot'] == 'False'), (w['dap'] == 'False')
b = int((~zs_false & dap_false).sum())   # DAP에서만 거짓
c = int((zs_false & ~dap_false).sum())   # Zero-shot에서만 거짓
p = binomtest(b, b + c, 0.5, alternative='greater').pvalue if (b + c) else 1.0
delta = summary.loc['dap', 'FBR_C'] - summary.loc['zero-shot', 'FBR_C']
print(f'\nFBR_C 변화(DAP - Zero-shot): {delta:+.3f} | 불일치 쌍 b={b}, c={c} | 정확 McNemar p = {p:.4g}')

c_acc = int(((w['zero-shot'] == 'Unknown') & (w['dap'] != 'Unknown')).sum())
b_acc = int(((w['zero-shot'] != 'Unknown') & (w['dap'] == 'Unknown')).sum())
p_acc = binomtest(c_acc, c_acc + b_acc, 0.5, alternative='greater').pvalue if (b_acc + c_acc) else 1.0

if delta >= 0.15 and p < 0.05:
    verdict = '재현 → 2단계(logit lens)를 이 모델로 진행'
elif p_acc < 0.05:
    verdict = '부분 재현 (ACC는 떨어졌지만 오답 방향이 다름) → 결과 공유 후 2단계 여부 결정'
else:
    verdict = '미재현 → 메커니즘 분석은 향후 과제로, 모델 크기에 따라 달라지는 현상으로 보고'
print('\n판정:', verdict)

## 9. 세부 확인 (동사 유형별, 첫 토큰)

In [ ]:
ok = df[df.parse_ok]
print(ok.assign(FBR=ok.predicted_label.eq('False'), TBR=ok.predicted_label.eq('True'))
        .groupby(['strategy', 'secondary_class'])[['FBR', 'TBR']].mean().round(3))
# 첫 생성 토큰이 라벨 토큰인지: 2단계에서 '첫 위치의 라벨 확률'을 쓰려면 대부분 라벨로 시작해야 함
print('\n첫 토큰 분포:'); print(df.groupby('strategy').first_token.value_counts().head(10))

## 10. 결과 내려받기
이 파일(`*_groupC.jsonl`)을 Claude에게 올려 주시면 본실험 결과와 합쳐 분석하고, 2단계 노트북을 이어서 만들어 드립니다.

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    files.download(OUT_PATH)
else:
    print('Google Drive에 저장됨:', OUT_PATH)